# BirdCLEF 2026 — Generate Species Spectrograms V2 (Ranks 51+)

Generates mel-spectrogram PNGs for all species ranked 51 and above by training frequency, zips them in batches of 10, and uploads the result to the `species-v2-051-206` Kaggle dataset.

**Prerequisites:** Attach secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` to this notebook before running.

In [ ]:
from __future__ import annotations
import json
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

# --- spectrogram params (must match training notebook) ---
TARGET_SIZE = (224, 224)
SAMPLE_RATE = 32000
CLIP_DURATION_SECONDS = 5
CHUNK_OFFSETS_SECONDS = (0.0, 2.5)
BATCH_SIZE = 10

# --- paths ---
DATA_ROOT   = Path('/kaggle/input/birdclef-2026')
TRAIN_CSV   = DATA_ROOT / 'train.csv'
TRAIN_AUDIO = DATA_ROOT / 'train_audio'
WORK_ROOT   = Path('/kaggle/working/work')
ZIP_STAGING = Path('/kaggle/working/upload')
ERROR_LOG   = Path('/kaggle/working/errors.log')

# --- rank range: 51 through all remaining species ---
RANK_START = 51
RANK_END   = None  # None = process all species from RANK_START onward

# --- target Kaggle dataset ---
DATASET_ID    = 'ucheozoemena/species-v2-051-206'
DATASET_TITLE = 'BirdCLEF 2026 Species V2 Ranks 051-206'

In [ ]:
# Configure Kaggle API credentials from notebook secrets
from kaggle_secrets import UserSecretsClient

_s = UserSecretsClient()
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
kaggle_json = kaggle_dir / 'kaggle.json'
kaggle_json.write_text(json.dumps({
    'username': _s.get_secret('KAGGLE_USERNAME'),
    'key':      _s.get_secret('KAGGLE_KEY'),
}))
os.chmod(str(kaggle_json), 0o600)
print('Kaggle credentials configured.')

In [ ]:
# Helper functions
def get_chunks(samples, sample_rate):
    clip_length = CLIP_DURATION_SECONDS * sample_rate
    chunks = []
    for offset_secs in CHUNK_OFFSETS_SECONDS:
        start = int(round(offset_secs * sample_rate))
        while start + clip_length <= len(samples):
            chunks.append((
                int(round(offset_secs * 1000)),
                start // sample_rate,
                samples[start:start + clip_length],
            ))
            start += clip_length
    return chunks


def normalize_to_uint8(s_db):
    lo, hi = float(s_db.min()), float(s_db.max())
    if hi == lo:
        return np.zeros_like(s_db, dtype=np.uint8)
    return ((s_db - lo) / (hi - lo) * 255).astype(np.uint8)


def zip_dir(source_dir, zip_path):
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for fp in sorted(source_dir.rglob('*')):
            if fp.is_file():
                zf.write(fp, arcname=fp.relative_to(source_dir))

In [ ]:
# Setup directories and load species list
WORK_ROOT.mkdir(parents=True, exist_ok=True)
ZIP_STAGING.mkdir(parents=True, exist_ok=True)
if ERROR_LOG.exists():
    ERROR_LOG.unlink()

df = pd.read_csv(TRAIN_CSV)
species_ranked = df['primary_label'].value_counts().index.tolist()
target_species = species_ranked[RANK_START - 1:RANK_END]  # RANK_END=None → slice to end

print(f'Total species in dataset: {len(species_ranked)}')
print(f'Processing ranks {RANK_START}–{RANK_START + len(target_species) - 1}: {len(target_species)} species')
print(f'Audio files to process: {len(df[df["primary_label"].isin(target_species)])}')

In [ ]:
# Generate spectrograms batch by batch
for local_start in range(0, len(target_species), BATCH_SIZE):
    batch_species = target_species[local_start:local_start + BATCH_SIZE]
    global_start  = RANK_START - 1 + local_start          # 0-indexed
    global_end    = global_start + len(batch_species) - 1  # 0-indexed
    batch_id      = global_start // BATCH_SIZE + 1
    batch_slug    = f'batch_{batch_id:03d}_rank_{global_start + 1:03d}-{global_end + 1:03d}'

    batch_work = WORK_ROOT / batch_slug
    batch_zip  = ZIP_STAGING / f'{batch_slug}.zip'

    if batch_zip.exists():
        print(f'[{batch_slug}] zip exists, skipping.')
        continue

    if batch_work.exists():
        shutil.rmtree(batch_work)
    batch_work.mkdir(parents=True)

    batch_df = df[df['primary_label'].isin(batch_species)]
    generated, errors = 0, 0

    for row in tqdm(batch_df.itertuples(index=False), total=len(batch_df), desc=batch_slug, unit='file'):
        species_dir = batch_work / row.primary_label
        species_dir.mkdir(exist_ok=True)
        audio_path = TRAIN_AUDIO / row.filename
        try:
            samples, sr = librosa.load(audio_path, sr=SAMPLE_RATE)
            stem = Path(row.filename).stem
            for offset_ms, start_sec, chunk in get_chunks(samples, sr):
                out = species_dir / f'{stem}_o{offset_ms}ms_s{start_sec}.png'
                if out.exists():
                    continue
                s = librosa.feature.melspectrogram(
                    y=chunk, sr=SAMPLE_RATE,
                    n_mels=128, fmin=50, fmax=14000, n_fft=1024, hop_length=320,
                )
                Image.fromarray(
                    normalize_to_uint8(librosa.power_to_db(s, ref=np.max))
                ).resize(TARGET_SIZE).save(out)
                generated += 1
        except Exception as exc:
            errors += 1
            with ERROR_LOG.open('a') as f:
                f.write(f'{audio_path}\t{exc}\n')

    zip_dir(batch_work, batch_zip)
    mb = batch_zip.stat().st_size / 1e6
    shutil.rmtree(batch_work)
    print(f'[{batch_slug}] {generated} PNGs → {batch_zip.name} ({mb:.0f} MB), {errors} errors')

print('\nAll batches complete.')
if ERROR_LOG.exists():
    with ERROR_LOG.open() as f:
        lines = f.readlines()
    print(f'Total errors: {len(lines)} (see {ERROR_LOG})')

In [ ]:
# Upload to Kaggle
(ZIP_STAGING / 'dataset-metadata.json').write_text(json.dumps({
    'title': DATASET_TITLE,
    'id': DATASET_ID,
    'licenses': [{'name': 'CC0-1.0'}],
}, indent=2))

zips = sorted(ZIP_STAGING.glob('*.zip'))
total_mb = sum(z.stat().st_size for z in zips) / 1e6
print(f'Uploading {len(zips)} zip(s), {total_mb:.0f} MB total:')
for z in zips:
    print(f'  {z.name} ({z.stat().st_size / 1e6:.0f} MB)')

print('\nRunning: kaggle datasets version ...')
result = subprocess.run(
    ['kaggle', 'datasets', 'version', '-p', str(ZIP_STAGING), '-m', 'initial upload'],
    capture_output=True, text=True,
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)

if result.returncode != 0:
    print('version failed — trying create instead...')
    result = subprocess.run(
        ['kaggle', 'datasets', 'create', '-p', str(ZIP_STAGING)],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.stderr:
        print('STDERR:', result.stderr)

if result.returncode == 0:
    print('Upload complete!')
else:
    print(f'Upload failed with exit code {result.returncode}')